# Lesson 3.2 — Control Flow: if, with, range

**Phase 3: Advanced Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Write conditional template blocks with `if`, `else if`, `else`
- Use `with` to change scope and avoid repetitive paths
- Iterate over lists and maps with `range`
- Use the `and`, `or`, `not`, `eq`, `ne`, `gt`, `lt` comparison functions

---

## Setup

In [ ]:
rm -rf /tmp/controldemo && mkdir -p /tmp/controldemo/templates

cat > /tmp/controldemo/Chart.yaml << 'EOF'
apiVersion: v2
name: controldemo
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/controldemo/values.yaml << 'EOF'
ingress:
  enabled: false
  host: chart-example.local
  tls: false

serviceAccount:
  create: true
  name: ""

env:
  - name: APP_ENV
    value: production
  - name: LOG_LEVEL
    value: info

extraLabels:
  team: platform
  tier: backend
  costCenter: eng-001

replicaCount: 2
EOF

echo "Setup complete"

---

## `if` / `else if` / `else`

Use `{{- if <condition> }}` to conditionally include YAML blocks. Helm treats these values as **false**: `false`, `0`, empty string `""`, `nil`, empty list `[]`, empty map `{}`.

In [ ]:
cat > /tmp/controldemo/templates/ingress.yaml << 'EOF'
{{- if .Values.ingress.enabled }}
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: {{ .Release.Name }}-ingress
spec:
  rules:
    - host: {{ .Values.ingress.host }}
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: {{ .Release.Name }}
                port:
                  number: 80
  {{- if .Values.ingress.tls }}
  tls:
    - hosts:
        - {{ .Values.ingress.host }}
      secretName: {{ .Release.Name }}-tls
  {{- end }}
{{- end }}
EOF

echo "=== ingress.enabled=false (should produce nothing) ==="
helm template my-release /tmp/controldemo --show-only templates/ingress.yaml

echo ""
echo "=== ingress.enabled=true (should produce Ingress) ==="
helm template my-release /tmp/controldemo \
  --show-only templates/ingress.yaml \
  --set ingress.enabled=true \
  --set ingress.host=myapp.example.com

### Comparison Functions

In [ ]:
cat > /tmp/controldemo/templates/comparison-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: comparison-demo
data:
  # eq: equal
  {{- if eq .Values.replicaCount 1 }}
  mode: single-replica
  {{- else if gt .Values.replicaCount 3 }}
  mode: high-availability
  {{- else }}
  mode: standard
  {{- end }}

  # and / or / not
  {{- if and .Values.ingress.enabled .Values.ingress.tls }}
  tlsEnabled: "true"
  {{- end }}
EOF

echo "=== replicaCount=1 ==="
helm template my-release /tmp/controldemo --show-only templates/comparison-demo.yaml --set replicaCount=1

echo ""
echo "=== replicaCount=5 ==="
helm template my-release /tmp/controldemo --show-only templates/comparison-demo.yaml --set replicaCount=5

echo ""
echo "=== replicaCount=2 (default) ==="
helm template my-release /tmp/controldemo --show-only templates/comparison-demo.yaml

---

## `with` — Narrow Scope

`with` is like `if` but also changes the scope (`.`) to the specified value, reducing repetition:

In [ ]:
cat > /tmp/controldemo/templates/with-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: with-demo
  labels:
    # Without 'with' — repetitive:
    # team: {{ .Values.extraLabels.team }}
    # tier: {{ .Values.extraLabels.tier }}

    # With 'with' — . becomes .Values.extraLabels inside the block
    {{- with .Values.extraLabels }}
    team: {{ .team }}
    tier: {{ .tier }}
    {{- end }}
data:
  # Accessing parent scope with $ inside a 'with' block
  {{- with .Values.ingress }}
  ingressEnabled: {{ .enabled | quote }}
  ingressHost: {{ .host | quote }}
  # $ always refers to the top-level scope
  releaseName: {{ $.Release.Name | quote }}
  {{- end }}
EOF

helm template my-release /tmp/controldemo --show-only templates/with-demo.yaml

> **Important:** Inside `with` and `range` blocks, `$` always refers to the global root scope. Use `$.Values.x` or `$.Release.Name` to access top-level data.

---

## `range` — Iterate Over Lists and Maps

In [ ]:
cat > /tmp/controldemo/templates/range-demo.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: range-demo
data:
  # Iterate over a list of env var objects
  # Each item in the list becomes '.' inside the loop
  envVars: |
    {{- range .Values.env }}
    - {{ .name }}={{ .value }}
    {{- end }}

  # Iterate over a map — $key and $value are loop variables
  labels: |
    {{- range $key, $value := .Values.extraLabels }}
    {{ $key }}: {{ $value }}
    {{- end }}
EOF

helm template my-release /tmp/controldemo --show-only templates/range-demo.yaml

In [ ]:
# Practical use: generate container env vars from values
cat > /tmp/controldemo/templates/env-demo.yaml << 'EOF'
apiVersion: v1
kind: Pod
metadata:
  name: env-demo-pod
spec:
  containers:
    - name: app
      image: nginx
      env:
        {{- range .Values.env }}
        - name: {{ .name | quote }}
          value: {{ .value | quote }}
        {{- end }}
EOF

helm template my-release /tmp/controldemo --show-only templates/env-demo.yaml

---

## ServiceAccount Conditional Pattern

This is a very common real-world pattern — create a ServiceAccount only when requested:

In [ ]:
cat > /tmp/controldemo/templates/serviceaccount.yaml << 'EOF'
{{- if .Values.serviceAccount.create -}}
apiVersion: v1
kind: ServiceAccount
metadata:
  name: {{ .Values.serviceAccount.name | default (printf "%s-%s" .Release.Name .Chart.Name) }}
  namespace: {{ .Release.Namespace }}
{{- end }}
EOF

echo "=== create=true (default) ==="
helm template my-release /tmp/controldemo --show-only templates/serviceaccount.yaml

echo ""
echo "=== create=false (nothing rendered) ==="
helm template my-release /tmp/controldemo \
  --show-only templates/serviceaccount.yaml \
  --set serviceAccount.create=false

---

## Exercise

In [ ]:
# TASK 1: Add a values key 'podAnnotations: {}' and write a template
#         that uses 'with' to only emit the annotations block if podAnnotations is non-empty.
#         Test with: --set 'podAnnotations.prometheus\.io/scrape=true'
# TODO:


# TASK 2: Add a list 'ports: [{name: http, port: 80}, {name: metrics, port: 9090}]'
#         to values.yaml, then use range to generate a Service with multiple ports.
# TODO:


# TASK 3: Write an if/else that sets a ConfigMap data value:
#         - 'logFormat: json'   when environment == 'production'
#         - 'logFormat: text'   otherwise
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

| Construct | Use Case |
|---|---|
| `{{- if .Values.x }}` | Conditionally include a block |
| `{{- if eq .Values.x "val" }}` | Equality check |
| `{{- if and .Values.a .Values.b }}` | Logical AND |
| `{{- with .Values.x }}` | Narrow scope to `.Values.x`; skips if empty |
| `$` | Always the root scope inside `with`/`range` |
| `{{- range .Values.list }}` | Iterate over a list |
| `{{- range $k, $v := .Values.map }}` | Iterate over a map |

---

## Next Lesson

➡️ [Lesson 3.3 — Named Templates & Helpers](lesson-3.3-helpers-and-partials.ipynb)